In [5]:
!git clone -b robs-branch --single-branch https://github.com/RobertDavidson1/GrokkingExperiments.git
%cd /content/GrokkingExperiments

%pip install --no-deps -e .
%pip install --no-deps scienceplots

Cloning into 'GrokkingExperiments'...
remote: Enumerating objects: 420, done.
remote: Counting objects: 100% (420/420), done.
remote: Compressing objects: 100% (199/199), done.
remote: Total 420 (delta 186), reused 363 (delta 140), pack-reused 0 (from 0)
Receiving objects: 100% (420/420), 20.59 MiB | 6.57 MiB/s, done.
Resolving deltas: 100% (186/186), done.
/content/GrokkingExperiments
Obtaining file:///content/GrokkingExperiments
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for grokkingexperiments (pyproject.toml) ... done
  Created wheel for grokkingexperiments: filename=grokkingexperiments-0.1.0-py3-none-any.whl size=3482 sha256=d28ee81ddde10c96945f4514b0b40cab567511e36922f488a188f0a2babff7e9
  Stored in directory: /tmp/pip-ephem-wheel-cache-ttzl5et4/wheels/d3/e5/64/e6e5467af90300195e6acf9aa85f648301f26

In [6]:
from grokkingexperiments.config import print_config, load_config
from grokkingexperiments.model import GrokkingModel
from grokkingexperiments.groups.zn import Zn

import torch
import numpy as np
import matplotlib.pyplot as plt

import scienceplots
plt.style.use(["science", "ieee", "notebook"])
# plt.style.use(["science", "notebook"])


device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} to train")

Using cuda to train


In [9]:
config = load_config()
config.group.n  = 112
config.train.epochs = 250_000
config.settings.progress_interval = 10_000

print_config(config)




Config
  model
    | actv_func            relu
    | embed_dim            128
    | hidden_dim           256
  train
    | epochs               250000
  group
    | name                 Zn
    | n                    112
    | split                0.4
  settings
    | seed                 1337
    | print_progress       True
    | progress_interval    10000
    | model_dir            data/models/
    | model_filename       custom_model.pt


In [10]:
left_embed_weights = {}
right_embed_weights = {}

n_seeds = 30

for seed in range(n_seeds):
    print(f"Training model with seed {seed}")
    config.settings.seed = seed # Overwrite seed

    model = GrokkingModel(config).to(device)
    zn = Zn(config)

    # Generate the data
    x_train, y_train, x_test, y_test = zn.generate_train_test_data()

    # Move to GPU if available
    x_train = x_train.to(device)
    y_train = y_train.to(device)
    x_test = x_test.to(device)
    y_test = y_test.to(device)

    # Train the model
    model.fit(x_train, y_train, x_test, y_test)


    # Store both embeddings on CPU
    E_L = model.state_dict()["E_L"]
    left_embed_weights[seed] = E_L.detach().cpu().clone()

    E_R = model.state_dict()["E_R"]
    right_embed_weights[seed] = E_R.detach().cpu().clone()


embed_weights = {
    "left": left_embed_weights,
    "right": right_embed_weights
}

torch.save(embed_weights, "embed_weights.pt")

Training model with seed 0
Epoch  0      | Loss 4.721  | Test loss 4.722  | Train accuracy 0.008  | Test accuracy 0.007 
Epoch  10000  | Loss 0.000  | Test loss 0.000  | Train accuracy 1.000  | Test accuracy 1.000 


KeyboardInterrupt: 

In [11]:
from google.colab import files
files.download("embed_weights.pt")


embed_weights = torch.load(
    "embed_weights.pt",
    map_location="cpu",
    weights_only=True
)

NameError: name 'files' is not defined